In [10]:
import torch
import torch.nn.functional as F

words = open("names.txt", "r").read().splitlines()

In [11]:
words[:5], len(words)

(['emma', 'olivia', 'ava', 'isabella', 'sophia'], 32033)

In [12]:
word = enumerate(sorted(list(set("".join(words)))))

stoi = {c: i+1 for i, c in word}
stoi["."] = 0

itos = {i: c for c, i in stoi.items()}

In [13]:
block_size = 3
X, Y = [], []

for i in words:
    context = [0] * block_size
    for c in i + ".":
        ix = stoi[c]
        X.append(context)
        Y.append(ix)
        
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [14]:
C = torch.randn((27,2))
W1 = torch.randn((6,100))
b1 = torch.randn((1,100))
W2 = torch.randn((100, 27))
b2 = torch.randn((1,27))
emb = C[X]
parameters = [C, W1, b1, W2, b2]
for p in parameters: p.requires_grad = True

In [15]:
h = torch.tanh(emb.view(-1,6) @ W1 + b1)

In [16]:
logits = h @ W2 + b2

In [17]:
loss = F.cross_entropy(logits, Y)

In [18]:
for i in range(10000):
    #minibatch
    ix = torch.randint(0, X.shape[0], (32,))
    
    #forward pass
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1,6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])

    for p in parameters: p.grad = None

    loss.backward()

    for p in parameters: p.data += -0.1 * p.grad

print(loss)  

tensor(2.4228, grad_fn=<NllLossBackward0>)


In [24]:
g = torch.Generator().manual_seed(2487363647)

for i in range(10):
    out = []
    context = [0] * block_size

    while True:
        x = torch.tensor([context])
        
        #forward pass
        emb = C[x]
        h = torch.tanh(emb.view(-1,6) @ W1 + b1)
        logits = h @ W2 + b2

        counts = logits.exp()
        prob = counts / counts.sum(1, keepdims=True)

        ix = torch.multinomial(prob, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(itos[ix])

        if ix == 0:
            break

    print("".join(out))            

ger.
drsil.
srew.
zul.
diccon.
komerele.
rule.
draley.
lal.
kayla.


In [55]:
prob.shape

torch.Size([32, 27])